# Stochastic Gradient Descent (SGD) — Beginner Masterclass

**Main reading:** [GeeksforGeeks — Stochastic Gradient Descent (SGD)](https://www.geeksforgeeks.org/machine-learning/ml-stochastic-gradient-descent-sgd/)

This Colab-compatible notebook follows the article's example and explains every important step: creating synthetic linear data, writing the model, calculating the gradient, updating parameters one sample at a time, plotting the loss, and interpreting the fitted line.

It adds worked calculations, a pure-Python implementation, a NumPy implementation, learning-rate experiments, a fair comparison with batch and mini-batch gradient descent, a PyTorch version, debugging tips, and official documentation links.

## Learning outcomes

By the end, you will understand:
- What makes gradient descent **stochastic**.
- How SGD updates the slope and intercept from one training example.
- How to implement SGD from scratch without ML libraries.
- Why the data are shuffled every epoch.
- How learning rate and batch size change the loss curve.
- How SGD differs from full-batch and mini-batch gradient descent.
- How to check the implementation against PyTorch and scikit-learn.

## 1. What is Stochastic Gradient Descent?

**Stochastic Gradient Descent (SGD)** is an iterative optimization algorithm. Instead of calculating a gradient using the entire training dataset for every update, it uses one randomly chosen training example at a time.

The standard parameter update is:

\[
\boxed{\theta_{t+1}=\theta_t-\eta\nabla_\theta J_i(\theta_t)}
\]

where:
- \(\theta_t\): model parameters at update \(t\).
- \(\eta\): learning rate.
- \(J_i\): loss for the selected training example \(i\).
- \(\nabla_\theta J_i\): gradient of that example's loss.

Because each example gives only an estimate of the direction that lowers the full-dataset loss, individual updates are noisy. This can make SGD computationally efficient, but its loss curve may fluctuate.

**Terminology note:** Strictly, *SGD* uses one sample per update. Many libraries and deep-learning discussions use “SGD” more broadly for mini-batch gradient descent too. We'll separate the three variants explicitly below.

**Source:** [GeeksforGeeks — SGD: overview and update rule](https://www.geeksforgeeks.org/machine-learning/ml-stochastic-gradient-descent-sgd/)

## 2. The article's example: synthetic linear-regression data

The article generates 100 input values between 0 and 2 and creates targets approximately following:

\[
\boxed{y=4+3X+\text{noise}}
\]

The underlying values are:
- True intercept: \(4\).
- True slope: \(3\).
- Added random noise: normally distributed noise, making the data more realistic.

Our objective is to recover an approximate line from noisy examples. The data are created directly in the notebook, so no dataset upload or download is needed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import time

SEED = 42
rng = np.random.default_rng(SEED)

# Matches the article's data-generation idea: 100 samples in [0, 2].
X = 2 * rng.random((100, 1))
y = 4 + 3 * X + rng.standard_normal((100, 1))

print("X shape:", X.shape)
print("y shape:", y.shape)
print("First 5 X values:", X[:5].ravel())
print("First 5 y values:", y[:5].ravel())
print("Known generating intercept = 4")
print("Known generating slope = 3")

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.scatter(X[:, 0], y[:, 0], alpha=0.75, label="Generated data")
plt.xlabel("X")
plt.ylabel("y")
plt.title("Synthetic data: y = 4 + 3X + noise")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 3. Define the model and loss

For one input feature, linear regression predicts:

\[
\hat{y}_i=\theta_0+\theta_1x_i
\]

where \(\theta_0\) is the intercept (bias) and \(\theta_1\) is the slope (weight).

We use Mean Squared Error (MSE):

\[
J(\theta_0,\theta_1)=\frac{1}{N}\sum_{i=1}^{N}(\hat{y}_i-y_i)^2
\]

SGD will update parameters using the loss of one example. We still calculate the **full-dataset MSE after each epoch** to monitor overall progress.

That distinction matters: the update uses a single example, while the plotted epoch loss measures performance across all examples.

## 4. Derive the gradient for one sample

For one sample \((x_i,y_i)\), define the squared-error loss:

\[
J_i=(\theta_0+\theta_1x_i-y_i)^2
\]

Let the prediction error be:

\[
e_i=\hat y_i-y_i
\]

By the chain rule:

\[
\frac{\partial J_i}{\partial\theta_0}=2e_i
\]

\[
\frac{\partial J_i}{\partial\theta_1}=2e_ix_i
\]

Therefore the single-example SGD updates are:

\[
\boxed{\theta_0\leftarrow\theta_0-\eta(2e_i)}
\]

\[
\boxed{\theta_1\leftarrow\theta_1-\eta(2e_ix_i)}
\]

The example contributes one gradient and the parameters change **immediately**. The next example is processed using the updated parameters.

## 5. Calculate one SGD update by hand

Use a tiny example:

\[
x_i=1.5,\qquad y_i=8.2
\]

Start from:

\[
\theta_0=0,\qquad\theta_1=0
\]

Choose \(\eta=0.01\).

**Step 1 — Prediction**

\[
\hat y_i=\theta_0+\theta_1x_i=0
\]

**Step 2 — Error**

\[
e_i=\hat y_i-y_i=0-8.2=-8.2
\]

**Step 3 — Gradients**

\[
\frac{\partial J_i}{\partial\theta_0}=2(-8.2)=-16.4
\]

\[
\frac{\partial J_i}{\partial\theta_1}=2(-8.2)(1.5)=-24.6
\]

**Step 4 — Update**

\[
\theta_0=0-0.01(-16.4)=0.164
\]

\[
\theta_1=0-0.01(-24.6)=0.246
\]

This one update has moved both parameters toward making the prediction larger. A single step is not expected to solve the whole problem; the algorithm repeats this process across many examples.

In [ ]:
# Pure Python: a transparent single-sample update
x_i = 1.5
y_i = 8.2
theta0 = 0.0
theta1 = 0.0
learning_rate = 0.01

prediction = theta0 + theta1 * x_i
error = prediction - y_i
loss_i = error ** 2

grad_theta0 = 2 * error
grad_theta1 = 2 * error * x_i

theta0_new = theta0 - learning_rate * grad_theta0
theta1_new = theta1 - learning_rate * grad_theta1

print("Step 1: prediction =", prediction)
print("Step 2: error =", error)
print("Step 3: single-sample squared loss =", loss_i)
print("Step 4: gradient for intercept =", grad_theta0)
print("Step 5: gradient for slope =", grad_theta1)
print("Step 6: updated intercept =", theta0_new)
print("Step 7: updated slope =", theta1_new)

## 6. What do intercept and slope mean?

The model is:

\[
\hat y=\theta_0+\theta_1X.
\]

- The **intercept** \(\theta_0\) is the prediction when \(X=0\).
- The **slope** \(\theta_1\) is the predicted change in \(y\) for a one-unit increase in \(X\).

Because our synthetic dataset was generated with intercept 4 and slope 3, the estimates should be near those values, but noise and finite sample size mean they need not be exact.

## 7. Implement SGD using plain Python only

The next implementation deliberately uses lists and loops rather than NumPy or a machine-learning framework.

Each epoch:
1. Shuffle the sample indices.
2. Visit one sample at a time.
3. Predict using the current parameters.
4. Calculate that sample's error and gradients.
5. Update the parameters immediately.
6. Calculate full-dataset MSE after the epoch.

Shuffling changes the order in which samples influence the updates, which helps avoid repeatedly seeing examples in the same order.

In [ ]:
# Plain Python SGD. No NumPy used inside this function.
def sgd_plain_python(x_values, y_values, learning_rate=0.01, epochs=100, seed=42):
    if len(x_values) != len(y_values):
        raise ValueError("X and y must have the same number of samples.")
    n = len(x_values)
    theta0, theta1 = 0.0, 0.0
    loss_history = []
    rng_local = __import__("random").Random(seed)

    for epoch in range(epochs):
        indices = list(range(n))
        rng_local.shuffle(indices)

        for i in indices:
            x_i = x_values[i]
            y_i = y_values[i]
            y_hat = theta0 + theta1 * x_i
            error = y_hat - y_i

            grad0 = 2 * error
            grad1 = 2 * error * x_i

            theta0 -= learning_rate * grad0
            theta1 -= learning_rate * grad1

        # Full-dataset MSE for monitoring, not for the individual updates.
        mse = sum(
            (theta0 + theta1*x_values[i] - y_values[i])**2
            for i in range(n)
        ) / n
        loss_history.append(mse)

    return theta0, theta1, loss_history

x_list = X[:, 0].tolist()
y_list = y[:, 0].tolist()

theta0_py, theta1_py, losses_py = sgd_plain_python(
    x_list, y_list, learning_rate=0.01, epochs=100, seed=42
)

print("Plain Python SGD:")
print("Estimated intercept:", theta0_py)
print("Estimated slope:", theta1_py)
print("Initial monitored MSE:", losses_py[0])
print("Final monitored MSE:", losses_py[-1])

## 8. Implement SGD from scratch using NumPy

The article's approach adds a column of ones to the feature matrix to represent the intercept:

\[
X_{\text{bias}}=
\begin{bmatrix}
1&x_1\\
1&x_2\\
\vdots&\vdots\\
1&x_N
\end{bmatrix}
\]

The parameter vector is:

\[
\theta=
\begin{bmatrix}
\theta_0\\
\theta_1
\end{bmatrix}
\]

Predictions are:

\[
\hat y=X_{\text{bias}}\theta.
\]

For a mini-batch \(B\), the MSE gradient is:

\[
\nabla_\theta J_B=
\frac{2}{|B|}X_B^\top(X_B\theta-y_B)
\]

For strict SGD, \(|B|=1\). For a mini-batch, use the **actual size of the current batch** in the denominator; the last batch may be smaller than the requested batch size.

In [ ]:
def sgd_numpy(X, y, learning_rate=0.01, epochs=100, batch_size=1, seed=42):
    X = np.asarray(X, dtype=float).reshape(-1, 1)
    y = np.asarray(y, dtype=float).reshape(-1, 1)
    if len(X) != len(y):
        raise ValueError("X and y must have the same number of samples.")
    if batch_size < 1:
        raise ValueError("batch_size must be >= 1.")

    n = len(X)
    X_bias = np.c_[np.ones((n, 1)), X]
    rng_local = np.random.default_rng(seed)
    theta = np.zeros((2, 1), dtype=float)
    cost_history = []

    for epoch in range(epochs):
        indices = rng_local.permutation(n)
        X_shuffled = X_bias[indices]
        y_shuffled = y[indices]

        for start in range(0, n, batch_size):
            X_batch = X_shuffled[start:start + batch_size]
            y_batch = y_shuffled[start:start + batch_size]
            current_batch_size = len(X_batch)

            errors = X_batch @ theta - y_batch
            gradients = (2 / current_batch_size) * (X_batch.T @ errors)
            theta -= learning_rate * gradients

        predictions = X_bias @ theta
        cost = np.mean((predictions - y) ** 2)
        cost_history.append(cost)

    return theta, np.array(cost_history)

theta_sgd, cost_sgd = sgd_numpy(X, y, learning_rate=0.01, epochs=100, batch_size=1, seed=42)

print("NumPy SGD parameters [intercept, slope]:")
print(theta_sgd.ravel())
print("Initial epoch MSE:", cost_sgd[0])
print("Final epoch MSE:", cost_sgd[-1])

## 9. Plot the cost function and fitted regression line

The cost curve shows the full-dataset MSE recorded after each epoch. Since SGD makes noisy individual updates, the curve can wiggle even when its overall trend is downward.

The red fitted line uses the final estimated parameters; compare it with the blue data points and the underlying generating line \(y=4+3X\).

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, len(cost_sgd)+1), cost_sgd)
plt.xlabel("Epoch")
plt.ylabel("Full-dataset MSE")
plt.title("Cost function during SGD training")
plt.grid(alpha=0.25)
plt.show()

X_line = np.linspace(X.min(), X.max(), 200).reshape(-1, 1)
X_line_bias = np.c_[np.ones((len(X_line), 1)), X_line]
y_line_sgd = X_line_bias @ theta_sgd
y_line_true = 4 + 3 * X_line

plt.figure(figsize=(8, 4.5))
plt.scatter(X[:, 0], y[:, 0], alpha=0.65, label="Generated data")
plt.plot(X_line[:, 0], y_line_true[:, 0], linestyle="--", linewidth=2, label="True generating line")
plt.plot(X_line[:, 0], y_line_sgd[:, 0], linewidth=2, label="SGD fitted line")
plt.xlabel("X")
plt.ylabel("y")
plt.title("Linear regression learned by SGD")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

print(f"Estimated model: y = {theta_sgd[0,0]:.4f} + {theta_sgd[1,0]:.4f} X")
print("True model before noise: y = 4 + 3 X")

## 10. Why does the article shuffle data every epoch?

An **epoch** is one pass through the entire training set.

Without shuffling, samples are always processed in the same order. In some datasets, samples may be grouped by class, time, or target value; repeatedly using that order can create undesirable update patterns.

Shuffling means each epoch uses a new ordering while preserving the same training examples. It does not create new data; it changes the order in which the noisy updates are applied.

In [ ]:
rng_show = np.random.default_rng(123)
indices = np.arange(12)
order_epoch_1 = rng_show.permutation(indices)
order_epoch_2 = rng_show.permutation(indices)

print("Original sample indices:", indices)
print("Order in epoch 1:", order_epoch_1)
print("Order in epoch 2:", order_epoch_2)

## 11. Compare batch, stochastic, and mini-batch gradient descent

The difference is how many examples contribute to **one parameter update**:

| Variant | Samples used per update | Typical behavior |
|---|---:|---|
| Batch gradient descent | All \(N\) samples | Stable gradient, potentially costly update |
| Stochastic gradient descent | 1 sample | Cheap, noisy updates |
| Mini-batch gradient descent | A small batch, such as 16/32/64 | Common compromise between noise and efficient computation |

All three can use the same MSE objective. The difference is how the gradient is estimated at each update.

In [ ]:
variant_results = {}
for name, batch_size in [
    ("Batch (full dataset)", len(X)),
    ("Mini-batch (16)", 16),
    ("SGD (1 sample)", 1),
]:
    theta_v, loss_v = sgd_numpy(
        X, y, learning_rate=0.01, epochs=100,
        batch_size=batch_size, seed=42
    )
    variant_results[name] = (theta_v, loss_v)
    print(
        f"{name:>22}: intercept={theta_v[0,0]:.3f}, "
        f"slope={theta_v[1,0]:.3f}, final MSE={loss_v[-1]:.4f}"
    )

plt.figure(figsize=(9, 4.5))
for name, (_, losses) in variant_results.items():
    plt.plot(losses, label=name)
plt.xlabel("Epoch")
plt.ylabel("Full-dataset MSE")
plt.title("Batch vs mini-batch vs stochastic gradient descent")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 12. Learning rate experiments

The learning rate controls how far each gradient update moves the parameters.

- Too small: learning may be very slow.
- Suitable: loss often decreases efficiently.
- Too large: the model can overshoot, oscillate, or diverge.

For SGD, the effect can be especially visible because the individual-example gradients are noisy. We use the same generated dataset, random seed, batch size, and epoch count, varying only the learning rate.

In [ ]:
lr_results = {}
for lr in [0.001, 0.01, 0.05, 0.1]:
    theta_lr, loss_lr = sgd_numpy(
        X, y, learning_rate=lr, epochs=60, batch_size=1, seed=123
    )
    lr_results[lr] = (theta_lr, loss_lr)
    finite = np.all(np.isfinite(loss_lr))
    print(
        f"lr={lr:<6}: finite={finite}, final MSE="
        f"{loss_lr[-1] if finite else float('nan'):.4f}, "
        f"parameters={theta_lr.ravel()}"
    )

plt.figure(figsize=(9, 4.5))
for lr, (_, losses) in lr_results.items():
    safe_losses = np.where(np.isfinite(losses), losses, np.nan)
    plt.plot(safe_losses, label=f"lr={lr}")
plt.xlabel("Epoch")
plt.ylabel("Full-dataset MSE")
plt.title("SGD: impact of learning rate")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 13. A PyTorch version — same idea, framework handles gradients

In PyTorch, autograd calculates gradients; `torch.optim.SGD` applies the parameter updates. The training loop still has the same high-level steps: predict, calculate loss, backpropagate, and step the optimizer.

For a fair comparison, we use MSE and the same generated data. `DataLoader` shuffles examples and provides mini-batches. Setting `batch_size=1` gives strict SGD; setting a larger batch size gives mini-batch SGD.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32)

dataset = TensorDataset(X_t, y_t)
loader = DataLoader(dataset, batch_size=1, shuffle=True)

model = nn.Linear(1, 1)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

torch_losses = []
for epoch in range(100):
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad()
        predictions = model(xb)
        loss = criterion(predictions, yb)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        full_predictions = model(X_t)
        epoch_mse = criterion(full_predictions, y_t).item()
    torch_losses.append(epoch_mse)

print("PyTorch intercept:", model.bias.item())
print("PyTorch slope:", model.weight.item())
print("Final full-data MSE:", torch_losses[-1])

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(torch_losses)
plt.xlabel("Epoch")
plt.ylabel("Full-dataset MSE")
plt.title("PyTorch SGD training curve")
plt.grid(alpha=0.25)
plt.show()

## 14. Optional reference implementation: scikit-learn `SGDRegressor`

`SGDRegressor` is a production-oriented implementation for linear regression trained using stochastic gradient-based optimization. It also supports regularization, learning-rate schedules, shuffling, stopping criteria, and other options.

Its defaults differ from our simple scratch implementation (for example, the loss may include regularization), so treat it as an API example, not as an exact numerical comparison to the unregularized hand-coded MSE objective.

In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

sk_sgd = make_pipeline(
    StandardScaler(),
    SGDRegressor(
        loss="squared_error",
        penalty=None,
        max_iter=3000,
        tol=1e-5,
        learning_rate="constant",
        eta0=0.01,
        random_state=42
    )
)
sk_sgd.fit(X, y.ravel())
sk_predictions = sk_sgd.predict(X)
print("scikit-learn SGDRegressor MSE:", mean_squared_error(y.ravel(), sk_predictions))
print("This model scales X internally; its coefficient/intercept correspond to the scaled feature.")

## 15. Advantages, applications, and challenges

### Advantages highlighted by the article
- **Fast individual updates:** each update uses only one example or a small batch.
- **Lower memory requirement:** no need to compute a full-dataset gradient at every step.
- **Large-dataset friendly:** well suited to iterative learning with many examples.
- **Online-learning potential:** parameters can be updated as new examples arrive.
- **Noisy updates:** stochastic variation can sometimes help optimization move through saddle regions or away from certain poor areas, though this is not guaranteed.

### Applications mentioned by the article
- Training deep neural networks.
- NLP, including Word2Vec-style models and transformer training setups.
- Computer vision, such as classification, detection, and segmentation.
- Reinforcement learning, including policy-gradient approaches.

### Challenges
- Noisy updates cause loss fluctuations.
- Learning-rate selection matters: too high may diverge; too low may be slow.
- Individual updates are cheap, but more updates may be needed overall.
- Convergence behavior depends on loss, data, parameterization, learning-rate schedule, and other settings.

## 16. Common implementation mistakes

1. **Using the wrong batch-size divisor.** For a mini-batch, divide by `len(X_batch)`, not blindly by the configured `batch_size`. The last batch can be smaller.
2. **Updating only after a whole epoch.** That turns the method into batch gradient descent if the gradient uses the full dataset. SGD updates parameters after each sample.
3. **No shuffling.** Reusing the same order every epoch can create systematic update patterns.
4. **Learning rate too large.** Watch for exploding loss or non-finite parameters.
5. **Not separating update loss from monitoring loss.** The per-sample loss drives an SGD update; full-dataset loss is useful for epoch-level monitoring.
6. **Incorrect gradient sign.** If error is defined as prediction minus target, the gradient must follow that sign consistently.
7. **Assuming SGD's loss must decrease at every sample update.** A single update can increase the full-dataset loss. Look at the overall trend across epochs.
8. **Assuming it always finds a global optimum.** Guarantees depend on the objective and assumptions. Neural-network losses are generally non-convex.

## 17. Knowledge check

Try these before reading the answers:

1. What is updated after each sample in strict SGD?
2. Why are samples shuffled each epoch?
3. How does strict SGD differ from batch gradient descent?
4. If the learning rate is too large, what might happen?
5. For \(J_i=(\theta_0+\theta_1x_i-y_i)^2\), write the two gradients.
6. Why do we calculate full-dataset MSE after each epoch if SGD trains one sample at a time?
7. What is the main implementation reason to use `len(X_batch)` in the mini-batch gradient formula?

In [ ]:
print("1. The model parameters are updated immediately using the selected sample's gradient.")
print("2. Shuffling changes the update order and reduces dependence on a fixed ordering of examples.")
print("3. Strict SGD uses one example per update; batch GD uses all examples per update.")
print("4. It may overshoot, oscillate, or diverge.")
print("5. dJ/dtheta0 = 2*error; dJ/dtheta1 = 2*error*x_i.")
print("6. It provides a consistent view of performance over the entire training set.")
print("7. The final mini-batch can be smaller than the configured batch size.")

## 18. References and documentation

### Main source
- [GeeksforGeeks — Stochastic Gradient Descent (SGD)](https://www.geeksforgeeks.org/machine-learning/ml-stochastic-gradient-descent-sgd/) — overview, update rule, synthetic data generation, scratch implementation, cost visualization, learned parameters, applications, advantages, and challenges.

### Official documentation
- [scikit-learn `SGDRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDRegressor.html) — regression trained with stochastic gradient-based optimization; includes learning-rate schedules, regularization, shuffling, and stopping criteria.
- [PyTorch `torch.optim.SGD`](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD) — SGD, momentum, weight decay, and Nesterov options.
- [Keras SGD optimizer](https://keras.io/api/optimizers/sgd/) — learning rate, momentum, Nesterov, weight decay, and clipping.
- [NumPy `permutation`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.permutation.html) — random shuffling of sample indices.
- [NumPy `dot`](https://numpy.org/doc/stable/reference/generated/numpy.dot.html) — dot products used in gradient calculations.
- [scikit-learn Mean Squared Error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html) — regression metric.

This notebook paraphrases the article and adds hand calculations, visual explanations, controlled experiments, and implementation checks rather than copying the full webpage verbatim.